# Looking at how many steps each takes to solve each puzzle

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

with open("data/traces_dataset/nonograms/test.jsonl", "r") as f:
    json_dataset = [json.loads(line) for line in f.readlines()]
traces = [
    {"puzzle_idx": i, "method": m, **run}
    for i, p in enumerate(json_dataset)
    for m, d in p["traces"].items()
    for run in next(iter(d.values()))
]

df = pd.json_normalize(traces)
df.head()


In [ ]:
pivot = (df.groupby(["method", "stopping_reason"])["num_steps"]
           .mean()
           .unstack()
           .round(2))

cmap = plt.get_cmap("viridis")
vals = pivot.values.astype(float)
log_vals = np.log10(np.clip(vals, 1e-9, None))
lo, hi = np.nanmin(log_vals), np.nanmax(log_vals)

def color_cell(v):
    if np.isnan(v):
        return ""
    t = (np.log10(max(v, 1e-9)) - lo) / (hi - lo)
    r, g, b, _ = cmap(t)
    return f"background-color: rgba({r*255:.0f},{g*255:.0f},{b*255:.0f},0.8)"

# pandas >= 2.1: use .map ; older: .applymap
pivot.style.map(color_cell).format("{:.1f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

solved = df[df['stopping_reason'] == 'SOLVED']

fig, ax = plt.subplots()

min_val = solved['num_steps'].min()
max_val = solved['num_steps'].max()
bins = np.logspace(np.log10(min_val), np.log10(max_val), 30)
bin_width_log = np.diff(np.log10(bins))[0]  # roughly constant bin width in log10 space

colors = plt.cm.tab10.colors

for i, (method, group) in enumerate(solved.groupby('method')):
    color = colors[i % len(colors)]
    vals = group['num_steps'].values

    # histogram
    counts, _, _ = ax.hist(vals, bins=bins, alpha=0.4, label=method, color=color)

    # fit normal to log10(num_steps)
    log_vals = np.log10(vals)
    mu, sigma = log_vals.mean(), log_vals.std()

    x_log = np.linspace(log_vals.min(), log_vals.max(), 200)
    pdf = norm.pdf(x_log, mu, sigma)

    # scale pdf to match histogram counts: N * bin_width_log
    scaled_pdf = pdf * len(vals) * bin_width_log

    x = 10 ** x_log
    ax.plot(x, scaled_pdf, color=color, linewidth=2)

ax.set_xscale('log')
ax.set_xlabel('num_steps (log scale)')
ax.set_ylabel('frequency')
ax.set_title('Frequency of num_steps for SOLVED runs, by method (with lognormal fits)')
ax.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

solved = df[df['stopping_reason'] == 'SOLVED']

fig, ax = plt.subplots()

# group data into a list of arrays, one per method
methods = sorted(solved['method'].unique())
data = [solved.loc[solved['method'] == m, 'num_steps'].values for m in methods]

ax.boxplot(data, labels=methods)
ax.set_yscale('log')
ax.set_ylabel('num_steps (log scale)')
ax.set_title('num_steps by method (SOLVED runs)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def mse(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return np.mean((a - b) ** 2)

# --- 1. Build long-format table: one row per (method, puzzle, step_index, mse) ---
records = []
for idx, row in df.iterrows():
    steps = row['steps']  # list of 2D grids
    for i in range(1, len(steps)):
        records.append({
            'method': row['method'],
            'puzzle_id': idx,          # or a real puzzle id column if you have one
            'step_index': i,           # 1st transition, 2nd transition, ...
            'mse': mse(steps[i], steps[i - 1]),
        })

long_df = pd.DataFrame(records)

def add_log_bins(long_df, n_bins=30):
    max_step = long_df['step_index'].max()
    bin_edges = np.unique(np.logspace(0, np.log10(max_step), n_bins).astype(int))
    long_df = long_df.copy()
    long_df['bin'] = pd.cut(long_df['step_index'], bins=bin_edges, include_lowest=True)
    # use bin midpoint (geometric mean) as the x-value for plotting
    long_df['bin_center'] = long_df['bin'].apply(
        lambda b: np.sqrt(b.left * b.right) if pd.notna(b) else np.nan
    )
    return long_df

long_df = add_log_bins(long_df, n_bins=30)

agg = (
    long_df.groupby(['method', 'bin_center'])['mse']
    .agg(['mean', 'count'])
    .reset_index()
    .dropna(subset=['bin_center'])
)

MIN_SAMPLES = 3
agg = agg[agg['count'] >= MIN_SAMPLES]

fig, ax = plt.subplots(figsize=(8, 5))
for method, sub in agg.groupby('method'):
    sub = sub.sort_values('bin_center')
    ax.plot(sub['bin_center'], sub['mean'], marker='o', markersize=3, label=method)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Step index (log scale)')
ax.set_ylabel('Mean MSE(step_t, step_{t-1})')
ax.legend()
plt.tight_layout()
plt.show()